# P6 · Failure analysis and trust in confidence
**Owner: Member A** · CMPS 261 Project 4.1 · Kepler Objects of Interest

The brief asks where the model makes mistakes (*weak signals, unusual stars, missing measurements, false positives that closely resemble real planets*) and whether its confidence can be trusted. Everything here uses the **final model's test predictions** at the threshold chosen on validation in P5. The test set is only *analysed*; no choice is made on it (thresholds for the mitigations are chosen on validation).

1. Error rate, missed planets and accepted false positives **by group**.
2. **False-positive types** (the `koi_fpflag_*` flags are used here only, never as model inputs).
3. **Hard cases**: the 10 false positives the model is most sure are planets.
4. **Confidence**: reliability plot and confident mistakes.
5. **Mitigations**: high-recall / high-precision thresholds, deferring uncertain cases to a human, recalibration.

## 0. Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/kepler-koi-classification/notebooks")

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             confusion_matrix, brier_score_loss)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from features import FEATURES, get_X

RESULTS = ROOT / "results"
FIGS = RESULTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#888888"
pd.set_option("display.max_columns", 30)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
print("Repo root:", ROOT)

## 1. Final model and threshold
Read from `results/final_choice.json` (written by B in P5). **If P5 is not pushed yet**, a stand-in applies the plan's rule to the validation predictions: for each model, take the threshold that maximises validation F1; pick the highest validation F1, but if a simpler model is within 0.01, take the simpler one. Rerun this notebook once `final_choice.json` exists so P6 analyses exactly the model B selected.

In [ ]:
MODELS = ["LogReg", "DecisionTree", "AdaBoost", "RandomForest", "NeuralNet"]   # simplest -> most complex
preds = {m: pd.read_csv(RESULTS / f"preds_{m}.csv") for m in MODELS if (RESULTS / f"preds_{m}.csv").exists()}

def best_threshold(y, p):
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, p >= t, zero_division=0) for t in grid]
    i = int(np.argmax(f1s))
    return grid[i], f1s[i]

choice_file = RESULTS / "final_choice.json"
if choice_file.exists():
    choice = json.loads(choice_file.read_text())
    FINAL = choice.get("model") or choice.get("final_model") or choice.get("name")
    THRESHOLD = float(choice.get("threshold") or choice.get("final_threshold"))
    SOURCE = "final_choice.json (P5)"
else:
    rows = []
    for m, d in preds.items():
        v = d[d["split"] == "val"]
        t, f1 = best_threshold(v["label"], v["p"])
        rows.append({"model": m, "val_threshold": t, "val_F1": round(f1, 4)})
    val_table = pd.DataFrame(rows).set_index("model")
    best_f1 = val_table["val_F1"].max()
    FINAL = next(m for m in MODELS if m in val_table.index and val_table.loc[m, "val_F1"] >= best_f1 - 0.01)
    THRESHOLD = float(val_table.loc[FINAL, "val_threshold"])
    SOURCE = "stand-in rule (P5 not pushed yet)"
    display(val_table)

print(f"Final model: {FINAL}   threshold: {THRESHOLD}   source: {SOURCE}")

In [ ]:
split = pd.read_csv(ROOT / "data" / "processed" / "koi_split.csv", low_memory=False)
p_all = preds[FINAL][["kepoi_name", "split", "p"]]
val  = split.merge(p_all[p_all["split"] == "val"][["kepoi_name", "p"]],  on="kepoi_name")
test = split.merge(p_all[p_all["split"] == "test"][["kepoi_name", "p"]], on="kepoi_name").copy()

test["pred"] = (test["p"] >= THRESHOLD).astype(int)
test["error"] = (test["pred"] != test["label"]).astype(int)
test["missed_planet"] = ((test["label"] == 1) & (test["pred"] == 0)).astype(int)    # false negative
test["accepted_fp"]   = ((test["label"] == 0) & (test["pred"] == 1)).astype(int)    # false positive
test["n_missing"] = get_X(test).isna().sum(axis=1)                                  # missing among the 21 model features

y, yhat = test["label"], test["pred"]
tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
print(f"Test rows: {len(test)}  |  F1 {f1_score(y, yhat):.3f}  precision {precision_score(y, yhat):.3f}  "
      f"recall {recall_score(y, yhat):.3f}  accuracy {accuracy_score(y, yhat):.3f}")
print(f"Confusion matrix: TN {tn}  FP {fp}  FN {fn}  TP {tp}   ->  {fn} missed planets, {fp} accepted false positives")

## 2. Where does it fail? Error rate by group
For each group: **error rate** (all mistakes / rows), **miss rate** (missed planets / real planets) and **false-alarm rate** (accepted false positives / real false positives). Groups follow the plan; SNR, transits, star and planet values are the raw catalogue values (not log-transformed).

In [ ]:
def bucket(s, bins, labels):
    out = pd.cut(s, bins=bins, labels=labels, right=False).astype(object)
    return out.where(s.notna(), "missing")

groupings = {
    "SNR":               bucket(test["koi_model_snr"], [-np.inf, 10, 50, np.inf], ["< 10", "10-50", "> 50"]),
    "transits observed": bucket(test["koi_num_transits"], [-np.inf, 4, np.inf], ["<= 3", "> 3"]),
    "missing values":    np.where(test["n_missing"] == 0, "0", ">= 1"),
    "star type":         np.where(test["koi_slogg"].isna(), "missing",
                                  np.where(test["koi_slogg"] < 3.5, "giant (logg < 3.5)", "dwarf (logg >= 3.5)")),
    "star temperature":  bucket(test["koi_steff"], [-np.inf, 5200, 6000, np.inf],
                                ["cool (< 5200 K)", "Sun-like (5200-6000 K)", "hot (> 6000 K)"]),
    "planet radius":     bucket(test["koi_prad"], [-np.inf, 15, np.inf], ["<= 15 Earth radii", "> 15 Earth radii"]),
}

rows = []
for gname, g in groupings.items():
    for level, d in test.groupby(np.asarray(g)):
        planets, fps = (d["label"] == 1).sum(), (d["label"] == 0).sum()
        rows.append({
            "group": gname, "level": level, "rows": len(d), "planet share": round(d["label"].mean(), 2),
            "error rate": round(d["error"].mean(), 3),
            "missed planets": int(d["missed_planet"].sum()),
            "miss rate": round(d["missed_planet"].sum() / planets, 3) if planets else np.nan,
            "accepted FPs": int(d["accepted_fp"].sum()),
            "false-alarm rate": round(d["accepted_fp"].sum() / fps, 3) if fps else np.nan,
        })
by_group = pd.DataFrame(rows).set_index(["group", "level"])
by_group.to_csv(RESULTS / "P6_errors_by_group.csv")
print(f"Overall test error rate: {test['error'].mean():.3f}")
by_group

In [ ]:
# Error rate per group (poster figure). Groups with < 20 test rows are flagged: their rates are noisy.
plot_df = by_group.reset_index()
plot_df["name"] = plot_df["group"] + ": " + plot_df["level"].astype(str)
plot_df = plot_df.iloc[::-1]
overall = test["error"].mean()

fig, ax = plt.subplots(figsize=(8, 0.38 * len(plot_df) + 1))
ax.barh(plot_df["name"], plot_df["error rate"], color=BLUE, height=0.6)
ax.axvline(overall, color=GRAY, lw=1, ls="--")
ax.text(overall, len(plot_df) - 0.4, f" overall {overall:.3f}", color="#555555", fontsize=9, va="bottom")
for i, (r, n) in enumerate(zip(plot_df["error rate"], plot_df["rows"])):
    ax.text(r, i, f"  {r:.3f}  (n={n}{', few rows' if n < 20 else ''})", va="center", fontsize=8, color="#333333")
ax.set_xlabel("test error rate")
ax.set_title(f"Where the final model ({FINAL}) makes mistakes", loc="left")
ax.set_xlim(0, max(plot_df["error rate"].max() * 1.45, overall * 2))
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGS / "P6_error_by_group.png", dpi=150)
plt.show()

## 3. Which kinds of false positive fool the model?
Among the **real false positives** in the test set: for each Kepler false-positive flag, how many carry it and how many the model wrongly called a planet. An object can carry several flags. (The flags are produced by vetting, so they are used **only here**, never as inputs.)

In [ ]:
FLAGS = {"koi_fpflag_nt": "not transit-like", "koi_fpflag_ss": "stellar eclipse",
         "koi_fpflag_co": "centroid offset", "koi_fpflag_ec": "ephemeris match (contamination)"}
fp_test = test[test["label"] == 0]
rows = []
for col, meaning in FLAGS.items():
    has = fp_test[fp_test[col] > 0]
    rows.append({"flag": col, "meaning": meaning, "false positives with flag": len(has),
                 "called planet": int(has["pred"].sum()),
                 "% called planet": round(100 * has["pred"].mean(), 1) if len(has) else np.nan})
no_flag = fp_test[(fp_test[list(FLAGS)] > 0).sum(axis=1) == 0]
rows.append({"flag": "(none)", "meaning": "no flag set", "false positives with flag": len(no_flag),
             "called planet": int(no_flag["pred"].sum()),
             "% called planet": round(100 * no_flag["pred"].mean(), 1) if len(no_flag) else np.nan})
fp_types = pd.DataFrame(rows).set_index("flag")
fp_types.to_csv(RESULTS / "P6_fp_types.csv")
fp_types

## 4. Hard cases: the 10 false positives with the highest probability
Compared with the median real planet in the test set. What do they share?

In [ ]:
SHOW = ["koi_period", "koi_depth", "koi_prad", "koi_model_snr", "koi_num_transits", "koi_steff", "koi_slogg",
        "koi_dicco_msky", "koi_dikco_msky", "koi_bin_oedp_sig", "n_missing"]
hard = (fp_test.sort_values("p", ascending=False).head(10)
               .set_index("kepoi_name")[["p"] + SHOW + list(FLAGS)])
compare = pd.DataFrame({
    "10 hardest FPs (median)": hard[SHOW].median(),
    "real planets (median)":   test.loc[test["label"] == 1, SHOW].median(),
    "all false positives (median)": fp_test[SHOW].median(),
}).round(3)
hard.round(3).to_csv(RESULTS / "P6_hard_cases.csv")
display(hard.round(3))
compare

## 5. Can the confidence be trusted? Reliability plot
Test predictions are put into 10 probability bins. In each bin: average predicted probability vs the real fraction of planets. A perfectly calibrated model lies on the diagonal. **Confident mistakes**: false positives with p > 0.9 and real planets with p < 0.1.

In [ ]:
def reliability(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            rows.append({"bin": f"{bins[b]:.1f}-{bins[b+1]:.1f}", "rows": int(m.sum()),
                         "mean predicted p": p[m].mean(), "fraction planets": y[m].mean()})
    t = pd.DataFrame(rows)
    ece = (t["rows"] * (t["mean predicted p"] - t["fraction planets"]).abs()).sum() / len(y)
    return t, ece

y_te, p_te = test["label"].to_numpy(), test["p"].to_numpy()
rel, ece = reliability(y_te, p_te)
conf_fp = int(((p_te > 0.9) & (y_te == 0)).sum())
conf_fn = int(((p_te < 0.1) & (y_te == 1)).sum())
print(f"Expected calibration error (ECE): {ece:.3f}   |   Brier score: {brier_score_loss(y_te, p_te):.4f}")
print(f"Confident mistakes: {conf_fp} false positives with p > 0.9, {conf_fn} real planets with p < 0.1")
rel.round(3)

## 6. Mitigations (measured on test, chosen on validation)
**(1) Threshold for the use case.** High recall = miss as few planets as possible (survey follow-up); high precision = only send objects that are very likely planets (expensive telescope time). Each threshold is chosen on **validation** to reach recall ≥ 0.98 or precision ≥ 0.98, then measured once on test.

**(2) Defer to a human** when 0.4 < p < 0.6: what share is deferred, and how good is the model on the rest?

**(3) Recalibration**: a one-feature logistic regression on the validation probabilities (Platt scaling, on the log-odds of p), applied to test; reliability plot redone. It changes the probabilities, not their order, so the F1 at the matching threshold is unchanged; the goal is probabilities that mean what they say.

In [ ]:
y_va, p_va = val["label"].to_numpy(), val["p"].to_numpy()
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)

def metrics(y, yhat):
    return {"precision": precision_score(y, yhat, zero_division=0), "recall": recall_score(y, yhat, zero_division=0),
            "F1": f1_score(y, yhat, zero_division=0), "accuracy": accuracy_score(y, yhat)}

# (1) thresholds chosen on validation
rec_ok  = [t for t in grid if recall_score(y_va, p_va >= t) >= 0.98]
prec_ok = [t for t in grid if precision_score(y_va, p_va >= t, zero_division=0) >= 0.98]
t_recall = max(rec_ok) if rec_ok else grid[0]          # highest threshold that still keeps recall >= 0.98
t_prec   = min(prec_ok) if prec_ok else grid[-1]       # lowest threshold that reaches precision >= 0.98

rows = []
for name, t in [("P5 threshold (max F1)", THRESHOLD), ("high recall (val recall >= 0.98)", t_recall),
                ("high precision (val precision >= 0.98)", t_prec)]:
    yhat = (p_te >= t).astype(int)
    rows.append({"setting": name, "threshold": t, **metrics(y_te, yhat),
                 "missed planets": int(((y_te == 1) & (yhat == 0)).sum()),
                 "accepted FPs": int(((y_te == 0) & (yhat == 1)).sum())})
thr_table = pd.DataFrame(rows).set_index("setting").round(3)
thr_table

In [ ]:
# (2) defer uncertain cases to a human
uncertain = (p_te > 0.4) & (p_te < 0.6)
auto = ~uncertain
yhat_all = (p_te >= THRESHOLD).astype(int)
defer = pd.DataFrame({
    "all test rows":         {"rows": len(y_te), **metrics(y_te, yhat_all)},
    "automatic (p outside 0.4-0.6)": {"rows": int(auto.sum()), **metrics(y_te[auto], yhat_all[auto])},
    "deferred (0.4 < p < 0.6)":      {"rows": int(uncertain.sum()), **metrics(y_te[uncertain], yhat_all[uncertain])},
}).T.round(3)
print(f"Deferred to a human: {uncertain.sum()} of {len(y_te)} test objects ({100 * uncertain.mean():.1f}%)")
defer

In [ ]:
# (3) recalibration: Platt scaling fitted on VALIDATION probabilities, applied to test
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p)).reshape(-1, 1)

calib = LogisticRegression().fit(logit(p_va), y_va)
p_te_cal = calib.predict_proba(logit(p_te))[:, 1]
rel_cal, ece_cal = reliability(y_te, p_te_cal)

print(f"ECE:   before {ece:.3f}  ->  after {ece_cal:.3f}")
print(f"Brier: before {brier_score_loss(y_te, p_te):.4f}  ->  after {brier_score_loss(y_te, p_te_cal):.4f}")
t_cal = float(calib.predict_proba(logit(np.array([THRESHOLD])))[:, 1][0])
print(f"The P5 threshold {THRESHOLD} corresponds to {t_cal:.3f} on the recalibrated scale "
      f"(same decisions, F1 unchanged: {f1_score(y_te, p_te_cal >= t_cal):.3f})")
conf_fp_cal = int(((p_te_cal > 0.9) & (y_te == 0)).sum()); conf_fn_cal = int(((p_te_cal < 0.1) & (y_te == 1)).sum())
print(f"Confident mistakes after recalibration: {conf_fp_cal} FPs with p > 0.9, {conf_fn_cal} planets with p < 0.1")

If the model's raw probabilities are squeezed into a narrow range (common for boosted trees), "0 confident mistakes" before recalibration only means it never says p > 0.9, and the 0.4-0.6 band catches many objects. Deferring on the **recalibrated** probabilities, where p means "the real chance of a planet", is the fairer version of mitigation (2):

In [ ]:
unc_cal = (p_te_cal > 0.4) & (p_te_cal < 0.6)
auto_cal = ~unc_cal
yhat_cal = (p_te_cal >= t_cal).astype(int)
defer_cal = pd.DataFrame({
    "automatic (calibrated p outside 0.4-0.6)": {"rows": int(auto_cal.sum()), **metrics(y_te[auto_cal], yhat_cal[auto_cal])},
    "deferred (0.4 < calibrated p < 0.6)":      {"rows": int(unc_cal.sum()), **metrics(y_te[unc_cal], yhat_cal[unc_cal])},
}).T.round(3)
print(f"Deferred on the calibrated scale: {unc_cal.sum()} of {len(y_te)} ({100 * unc_cal.mean():.1f}%)")
defer_cal

In [ ]:
# Reliability plot: before vs after recalibration (point size = number of test rows in the bin)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], color=GRAY, lw=1, ls="--", label="perfect calibration")
for t, color, name in [(rel, BLUE, f"before (ECE {ece:.3f})"), (rel_cal, ORANGE, f"after recalibration (ECE {ece_cal:.3f})")]:
    ax.plot(t["mean predicted p"], t["fraction planets"], color=color, lw=2, label=name, zorder=2)
    ax.scatter(t["mean predicted p"], t["fraction planets"], s=20 + 180 * t["rows"] / t["rows"].max(),
               color=color, edgecolor="white", linewidth=2, zorder=3)
ax.set_xlabel("mean predicted probability of planet")
ax.set_ylabel("actual fraction of planets")
ax.set_title(f"Reliability on the test set ({FINAL})", loc="left")
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.04)
ax.grid(alpha=0.25)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(FIGS / "P6_reliability.png", dpi=150)
plt.show()

In [ ]:
summary = {
    "final_model": FINAL, "threshold": THRESHOLD, "source": SOURCE,
    "test_F1": round(f1_score(y_te, yhat_all), 4), "missed_planets": int(fn), "accepted_fps": int(fp),
    "ECE_before": round(ece, 4), "ECE_after": round(ece_cal, 4),
    "confident_mistakes": {"fp_p_gt_0.9": conf_fp, "planet_p_lt_0.1": conf_fn},
    "deferred_share": round(float(uncertain.mean()), 4),
    "auto_F1": round(f1_score(y_te[auto], yhat_all[auto]), 4),
    "deferred_share_calibrated": round(float(unc_cal.mean()), 4),
    "auto_F1_calibrated": round(f1_score(y_te[auto_cal], yhat_cal[auto_cal]), 4),
    "high_recall_threshold": float(t_recall), "high_precision_threshold": float(t_prec),
}
(RESULTS / "P6_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved results/P6_*.csv, results/P6_summary.json, results/figures/P6_*.png")
print("Next: Ctrl+S, then commit and push → message 'P6: failure analysis'")

## Explain (for the report / professor)
*"We look for where the model breaks: weak signals, unusual stars, missing data and planet-like false positives, then test simple fixes."*

Points to write up from the outputs above:
- which groups have an error rate clearly above the overall one (check the row counts: small groups are noisy);
- which false-positive flag most often fools the model, and what the hardest false positives have in common;
- whether high-confidence predictions are reliable (reliability plot, confident mistakes), and what recalibration changed;
- the cost of each mitigation: planets missed vs false positives accepted, and the share of objects a human would have to check.